In [31]:
import os
import pickle
import warnings
import numpy as np
import pandas as pd
import xgboost as xgb

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

warnings.filterwarnings("ignore")

In [32]:
# 1) Load data
df = pd.read_csv("../data/processed_data.csv")
print(f"Data loaded: {df.shape}")

Data loaded: (69504, 31)


In [33]:
# 2) Encode city
le = None
if "city" in df.columns:
    le = LabelEncoder()
    df["city_encoded"] = le.fit_transform(df["city"])

In [34]:
# 3) Feature columns
base_features = [
    "temp", "humidity", "wind", "cloud", "rain",
    "month", "day", "pressure", "hour", "weekday", "lat", "lon"
]
feature_cols = [c for c in base_features if c in df.columns]
if "city_encoded" in df.columns:
    feature_cols = ["city_encoded"] + feature_cols

X = df[feature_cols]

In [50]:
# 4) Targets
targets = {
    "temp": ["temp_day1", "temp_day2", "temp_day3", "temp_day4"],
    "humidity": ["humidity_day1", "humidity_day2", "humidity_day3", "humidity_day4"],
    "wind": ["wind_day1", "wind_day2", "wind_day3", "wind_day4"],
    "rain": ["rain_day1", "rain_day2", "rain_day3", "rain_day4"],
}



In [51]:
metrics_summary = []
models = {}

for target_name, target_cols in targets.items():
    print(f"\n{'='*40}")
    print(f"Training: {target_name}")
    models[target_name] = []

    for i, col in enumerate(target_cols, start=1):
        if col not in df.columns:
            continue

        y = df[col]
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.2, random_state=42
        )

        if target_name in ["humidity", "rain"]:
            model = xgb.XGBRegressor(
                n_estimators=300, max_depth=8,
                learning_rate=0.05, random_state=42, n_jobs=-1
            )
        else:
            model = RandomForestRegressor(
                n_estimators=200, max_depth=15,
                random_state=42, n_jobs=-1
            )

        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)

        mae = mean_absolute_error(y_test, y_pred)
        rmse = np.sqrt(mean_squared_error(y_test, y_pred))
        r2 = r2_score(y_test, y_pred)

        print(f"  Day {i} — MAE: {mae:.3f} | RMSE: {rmse:.3f} | R2: {r2:.4f}")

        metrics_summary.append({
            "target": target_name,
            "day": i,
            "mae": round(mae, 3),
            "rmse": round(rmse, 3),
            "r2": round(r2, 4)
        })
        models[target_name].append(model)

metrics_df = pd.DataFrame(metrics_summary)
print("\n\nFull Metrics:")
print(metrics_df.to_string())


Training: temp
  Day 1 — MAE: 0.935 | RMSE: 1.325 | R2: 0.9481
  Day 2 — MAE: 1.064 | RMSE: 1.484 | R2: 0.9347
  Day 3 — MAE: 1.077 | RMSE: 1.493 | R2: 0.9334
  Day 4 — MAE: 1.059 | RMSE: 1.462 | R2: 0.9360

Training: humidity
  Day 1 — MAE: 4.806 | RMSE: 6.552 | R2: 0.9119
  Day 2 — MAE: 5.040 | RMSE: 6.809 | R2: 0.9047
  Day 3 — MAE: 5.073 | RMSE: 6.808 | R2: 0.9048
  Day 4 — MAE: 5.109 | RMSE: 6.836 | R2: 0.9037

Training: wind
  Day 1 — MAE: 2.113 | RMSE: 2.787 | R2: 0.6198
  Day 2 — MAE: 2.090 | RMSE: 2.733 | R2: 0.6393
  Day 3 — MAE: 1.974 | RMSE: 2.606 | R2: 0.6706
  Day 4 — MAE: 1.979 | RMSE: 2.602 | R2: 0.6678

Training: rain
  Day 1 — MAE: 0.210 | RMSE: 0.683 | R2: 0.2013
  Day 2 — MAE: 0.215 | RMSE: 0.706 | R2: 0.1763
  Day 3 — MAE: 0.219 | RMSE: 0.716 | R2: 0.1591
  Day 4 — MAE: 0.216 | RMSE: 0.677 | R2: 0.1512


Full Metrics:
      target  day    mae   rmse      r2
0       temp    1  0.935  1.325  0.9481
1       temp    2  1.064  1.484  0.9347
2       temp    3  1.077  1.

In [52]:
y_pred 

array([-0.00100428,  0.00596541, -0.01622546, ..., -0.00142726,
        0.0188938 ,  0.01771645], shape=(13901,), dtype=float32)

In [53]:
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

In [54]:
print(f"    MAE : {mae:.3f}")
print(f"    RMSE: {rmse:.3f}")
print(f"    R2  : {r2:.4f}")

    MAE : 0.216
    RMSE: 0.677
    R2  : 0.1512


In [56]:
# 6) Save
os.makedirs("../model", exist_ok=True)

with open("../model/model.pkl", "wb") as f:
    pickle.dump(models, f)

if le is not None:
    with open("../model/label_encoder.pkl", "wb") as f:
        pickle.dump(le, f)

metrics_df = pd.DataFrame(metrics_summary)
metrics_df.to_csv("../model/training_metrics.csv", index=False)

print("\n✅ Training complete!")
print("✅ model.pkl saved!")


✅ Training complete!
✅ model.pkl saved!
